# 05 — Race Result Prediction

Using features built from in-race telemetry, we train models to predict:

| Task | Type | Target |
|------|------|--------|
| Final position | Regression | `final_position` (1–20) |
| Top-10 finish | Classification | `top_10` (0/1) |
| Podium | Classification | `top_3` (0/1) |

**Models:** XGBoost · LightGBM  
**Validation:** Leave-One-Race-Out (LORO) — train on all races except one, test on that race.  
This mimics real-world use: predicting a race from prior races.

> **Prerequisite:** Run `04_feature_engineering.ipynb` first.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    accuracy_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
)
import xgboost as xgb
import lightgbm as lgb

sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print(f'XGBoost  : {xgb.__version__}')
print(f'LightGBM : {lgb.__version__}')

## 1 · Load Feature Matrix

In [ ]:
FEATURES_PATH = os.path.join('..', 'data', 'processed', 'features_2025.csv')
features = pd.read_csv(FEATURES_PATH)

print(f'Shape: {features.shape}')
print(f'Rounds: {sorted(features["RoundNumber"].dropna().astype(int).unique())}')
features.head(3)

## 2 · Prepare Features & Targets

In [ ]:
FEATURE_COLS = [
    'RoundNumber',
    'best_lap_sec',
    'average_lap_sec',
    'median_lap_sec',
    'std_lap_sec',
    'consistency',
    'pit_stops',
    'compound_changes',
    'fastest_sector1_sec',
    'fastest_sector2_sec',
    'fastest_sector3_sec',
    'avg_deg_slope',
    'pos_lap1',
    'is_wet_race',
    'team_encoded',  # created below
]

TARGET_REG   = 'final_position'
TARGET_TOP10 = 'top_10'
TARGET_TOP3  = 'top_3'

# Encode team as integer
le = LabelEncoder()
features['team_encoded'] = le.fit_transform(features['Team'].fillna('Unknown'))

# Drop rows where target is missing or placeholder (999)
model_df = features[
    features[TARGET_REG].notna() &
    (features[TARGET_REG] < 100)
].copy()

model_df[FEATURE_COLS] = model_df[FEATURE_COLS].apply(pd.to_numeric, errors='coerce')

print(f'Model-ready rows: {len(model_df)}')
print(f'Missing values per feature:')
model_df[FEATURE_COLS].isna().sum()[model_df[FEATURE_COLS].isna().sum() > 0]

In [ ]:
# Impute remaining NaNs with column median
for col in FEATURE_COLS:
    if model_df[col].isna().any():
        model_df[col] = model_df[col].fillna(model_df[col].median())

X = model_df[FEATURE_COLS].values
y_reg   = model_df[TARGET_REG].values.astype(float)
y_top10 = model_df[TARGET_TOP10].values.astype(int)
y_top3  = model_df[TARGET_TOP3].values.astype(int)
rounds  = model_df['RoundNumber'].values.astype(int)

print('Features ready:', X.shape)

## 3 · Leave-One-Race-Out Cross-Validation

For each round R: train on all other rounds, predict round R.  
This is the most realistic evaluation for a sequential prediction problem.

In [ ]:
unique_rounds = sorted(np.unique(rounds))
print(f'Rounds for LORO-CV: {unique_rounds}')
print(f'CV folds: {len(unique_rounds)}')

In [ ]:
# ── XGBoost regression (final position) ─────────────────────────────────────
xgb_reg_params = dict(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective='reg:squarederror', random_state=42, verbosity=0,
)

oof_reg = np.zeros(len(X))

for rnd in unique_rounds:
    train_mask = rounds != rnd
    test_mask  = rounds == rnd
    model = xgb.XGBRegressor(**xgb_reg_params)
    model.fit(X[train_mask], y_reg[train_mask])
    oof_reg[test_mask] = model.predict(X[test_mask])

mae  = mean_absolute_error(y_reg, oof_reg)
rmse = root_mean_squared_error(y_reg, oof_reg)
print(f'XGBoost Regression  — MAE: {mae:.2f}  RMSE: {rmse:.2f}')

In [ ]:
# ── LightGBM regression (comparison) ────────────────────────────────────────
lgb_reg_params = dict(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective='regression', random_state=42, verbose=-1,
)

oof_reg_lgb = np.zeros(len(X))

for rnd in unique_rounds:
    train_mask = rounds != rnd
    test_mask  = rounds == rnd
    model = lgb.LGBMRegressor(**lgb_reg_params)
    model.fit(X[train_mask], y_reg[train_mask])
    oof_reg_lgb[test_mask] = model.predict(X[test_mask])

mae_lgb  = mean_absolute_error(y_reg, oof_reg_lgb)
rmse_lgb = root_mean_squared_error(y_reg, oof_reg_lgb)
print(f'LightGBM Regression — MAE: {mae_lgb:.2f}  RMSE: {rmse_lgb:.2f}')

In [ ]:
# ── XGBoost Top-10 classification ───────────────────────────────────────────
xgb_cls_params = dict(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective='binary:logistic', eval_metric='logloss',
    random_state=42, verbosity=0,
)

oof_top10 = np.zeros(len(X))
oof_top10_proba = np.zeros(len(X))

for rnd in unique_rounds:
    train_mask = rounds != rnd
    test_mask  = rounds == rnd
    model = xgb.XGBClassifier(**xgb_cls_params)
    model.fit(X[train_mask], y_top10[train_mask])
    oof_top10[test_mask] = model.predict(X[test_mask])
    oof_top10_proba[test_mask] = model.predict_proba(X[test_mask])[:, 1]

acc_top10 = accuracy_score(y_top10, oof_top10)
auc_top10 = roc_auc_score(y_top10, oof_top10_proba)
print(f'Top-10 Classification — Accuracy: {acc_top10:.3f}  AUC: {auc_top10:.3f}')

In [ ]:
# ── XGBoost Podium classification ───────────────────────────────────────────
oof_top3 = np.zeros(len(X))
oof_top3_proba = np.zeros(len(X))

for rnd in unique_rounds:
    train_mask = rounds != rnd
    test_mask  = rounds == rnd
    model = xgb.XGBClassifier(**xgb_cls_params)
    model.fit(X[train_mask], y_top3[train_mask])
    oof_top3[test_mask] = model.predict(X[test_mask])
    oof_top3_proba[test_mask] = model.predict_proba(X[test_mask])[:, 1]

acc_top3 = accuracy_score(y_top3, oof_top3)
auc_top3 = roc_auc_score(y_top3, oof_top3_proba)
print(f'Podium Classification  — Accuracy: {acc_top3:.3f}  AUC: {auc_top3:.3f}')

## 4 · Results Summary

In [ ]:
results_summary = pd.DataFrame([
    {'Model': 'XGBoost',  'Task': 'Position (regression)', 'MAE': mae,     'RMSE': rmse,     'AUC': '-'},
    {'Model': 'LightGBM', 'Task': 'Position (regression)', 'MAE': mae_lgb, 'RMSE': rmse_lgb, 'AUC': '-'},
    {'Model': 'XGBoost',  'Task': 'Top-10 (classification)', 'MAE': '-', 'RMSE': '-', 'AUC': f'{auc_top10:.3f}'},
    {'Model': 'XGBoost',  'Task': 'Podium  (classification)', 'MAE': '-', 'RMSE': '-', 'AUC': f'{auc_top3:.3f}'},
])
print('Model Performance Summary (LORO-CV):')
results_summary

## 5 · Predicted vs Actual (Regression)

In [ ]:
pred_df = model_df[['EventName', 'Driver', 'Team', TARGET_REG]].copy()
pred_df['predicted_position_xgb'] = np.clip(np.round(oof_reg), 1, 20).astype(int)
pred_df['predicted_position_lgb'] = np.clip(np.round(oof_reg_lgb), 1, 20).astype(int)
pred_df['error_xgb'] = pred_df['predicted_position_xgb'] - pred_df[TARGET_REG]

fig = px.scatter(
    pred_df,
    x=TARGET_REG,
    y='predicted_position_xgb',
    color='Team',
    hover_data=['Driver', 'EventName'],
    title='XGBoost — Predicted vs Actual Final Position (LORO-CV)',
    labels={TARGET_REG: 'Actual Position', 'predicted_position_xgb': 'Predicted Position'},
    height=520,
    opacity=0.7,
)
# Perfect prediction line
fig.add_shape(type='line', x0=1, y0=1, x1=20, y1=20,
              line=dict(color='white', dash='dash', width=1))
fig.update_layout(legend_title='Constructor')
fig.show()

## 6 · Feature Importance (Full Dataset Model)

In [ ]:
# Retrain on all data to get stable feature importance
final_model = xgb.XGBRegressor(**xgb_reg_params)
final_model.fit(X, y_reg)

importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': final_model.feature_importances_,
}).sort_values('importance', ascending=True)

fig = px.bar(
    importance,
    x='importance',
    y='feature',
    orientation='h',
    title='XGBoost Feature Importance — Race Position Prediction',
    labels={'importance': 'Importance Score', 'feature': ''},
    color='importance',
    color_continuous_scale='Oranges',
    height=520,
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 7 · Confusion Matrix — Top-10 Classification

In [ ]:
print('Top-10 Classification Report:')
print(classification_report(y_top10, oof_top10, target_names=['Outside Top-10', 'Top-10']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_top10, oof_top10,
    display_labels=['Outside Top-10', 'Top-10'],
    colorbar=False, ax=ax, cmap='Oranges'
)
ax.set_title('Top-10 Confusion Matrix (LORO-CV)', fontsize=12)
plt.tight_layout()
plt.show()

## 8 · Best Predicted Race — Deep Dive

Find the race where the model performed best and show the full predicted vs actual grid.

In [ ]:
race_mae = (
    pred_df.groupby('EventName')
    .apply(lambda g: mean_absolute_error(g[TARGET_REG], g['predicted_position_xgb']))
    .reset_index(name='mae')
    .sort_values('mae')
)

best_race = race_mae.iloc[0]['EventName']
worst_race = race_mae.iloc[-1]['EventName']

print(f'Best predicted race  : {best_race}  (MAE: {race_mae.iloc[0]["mae"]:.2f})')
print(f'Worst predicted race : {worst_race}  (MAE: {race_mae.iloc[-1]["mae"]:.2f})')

In [ ]:
best_df = pred_df[pred_df['EventName'] == best_race].sort_values(TARGET_REG)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=best_df['Driver'], y=best_df[TARGET_REG],
    name='Actual', marker_color='#3671C6', opacity=0.8
))
fig.add_trace(go.Bar(
    x=best_df['Driver'], y=best_df['predicted_position_xgb'],
    name='Predicted', marker_color='#FF8000', opacity=0.8
))
fig.update_layout(
    barmode='group',
    title=f'Actual vs Predicted Position — {best_race}',
    xaxis_title='Driver', yaxis_title='Position',
    yaxis_autorange='reversed',
    height=460,
)
fig.show()

---
## ✅ Summary

Fill these in after running:

| Metric | Value |
|--------|-------|
| Position MAE (XGBoost) | — |
| Position MAE (LightGBM) | — |
| Top-10 AUC | — |
| Podium AUC | — |
| Most important feature | — |
| Best predicted race | — |